# 05 — Executive Summary

This notebook is designed to be read standalone. It loads every table and chart
produced by notebooks 01–04, assembles a single four-panel dashboard, and
synthesises the quantitative findings into a business-facing summary and four
concrete recommendations. No raw data processing happens here — everything is
derived from the saved outputs in `outputs/tables/` and `outputs/figures/`.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker

warnings.filterwarnings('ignore')

FIG_DIR = Path('../outputs/figures')
TBL_DIR = Path('../outputs/tables')

pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 60)

## 1. Load All Outputs

We start by scanning both output directories so we have a full inventory of
what was produced in the previous notebooks. Then we load every table into
memory and display a brief summary of each one. If a file is missing it means
the corresponding notebook has not been run yet — the dashboard and findings
sections will degrade gracefully using placeholder text.

In [ ]:
# Inventory of all saved outputs
all_tables  = sorted(TBL_DIR.glob('*.csv'))
all_figures = sorted(FIG_DIR.glob('*.png'))

print(f'Tables  : {len(all_tables)}')
for p in all_tables:
    print(f'  {p.name}')

print(f'\nFigures : {len(all_figures)}')
for p in all_figures:
    print(f'  {p.name}')

In [ ]:
def load_table(name):
    path = TBL_DIR / name
    if path.exists():
        return pd.read_csv(path)
    print(f'  [missing] {name}')
    return pd.DataFrame()

# Notebook 02 outputs
kpi_summary       = load_table('02_kpi_summary.csv')
monthly_kpis      = load_table('02_monthly_kpis.csv')
category_perf     = load_table('02_category_performance.csv')
review_delivery   = load_table('02_review_by_delivery_bin.csv')
order_freq        = load_table('02_order_frequency_dist.csv')

# Notebook 03 outputs
segment_summary   = load_table('03_segment_summary.csv')
cohort_retention  = load_table('03_cohort_retention_matrix.csv')
avg_retention     = load_table('03_avg_retention_curve.csv')
cat_repeat        = load_table('03_category_repeat_rate.csv')

# Notebook 04 outputs
power_analysis    = load_table('04_power_analysis.csv')
balance_check     = load_table('04_balance_check.csv')
ttest_results     = load_table('04_ttest_results.csv')
mw_results        = load_table('04_mann_whitney_results.csv')
reg_results       = load_table('04_regression_ate_results.csv')
ab_summary        = load_table('04_ab_summary.csv')
hist_clv_tiers    = load_table('04_historical_clv_tiers.csv')
rfm_clv_crosstab  = load_table('04_rfm_clv_crosstab.csv')
seg_clv_summary   = load_table('04_segment_clv_summary.csv')

print('Load complete.')

In [ ]:
# Shape summary of every loaded table
table_inventory = {
    '02_kpi_summary':         kpi_summary,
    '02_monthly_kpis':        monthly_kpis,
    '02_category_performance':category_perf,
    '02_review_by_delivery':  review_delivery,
    '03_segment_summary':     segment_summary,
    '03_cohort_retention':    cohort_retention,
    '03_avg_retention_curve': avg_retention,
    '03_category_repeat_rate':cat_repeat,
    '04_power_analysis':      power_analysis,
    '04_ttest_results':       ttest_results,
    '04_reg_ate_results':     reg_results,
    '04_historical_clv_tiers':hist_clv_tiers,
    '04_seg_clv_summary':     seg_clv_summary,
}

inv_df = pd.DataFrame(
    [(name, df.shape[0], df.shape[1], 'OK' if len(df) > 0 else 'MISSING')
     for name, df in table_inventory.items()],
    columns=['table', 'rows', 'cols', 'status'],
)
inv_df

In [ ]:
# Preview the four tables most referenced in the summary
for label, tbl in [
    ('KPI Summary',         kpi_summary),
    ('Segment Summary',     segment_summary),
    ('T-test Results',      ttest_results),
    ('Historical CLV Tiers',hist_clv_tiers),
]:
    if len(tbl) > 0:
        print(f'\n── {label} ──')
        display(tbl)

## 2. Four-Panel Executive Dashboard

This single image assembles the most informative chart from each of the four
analysis notebooks. Top-left shows the monthly revenue and order volume trends
from the EDA. Top-right shows the RFM customer segment treemap. Bottom-left
shows the cohort retention heatmap. Bottom-right shows the RFM segment cross-
validated against CLV tiers. Together they tell the full story: how the business
grew, who the customers are, how well they are retained, and how behavioural
segmentation aligns with monetary lifetime value.

In [ ]:
DASHBOARD_CHARTS = [
    (FIG_DIR / '02_monthly_kpis.png',             'NB02 — Monthly Revenue & Order Trends'),
    (FIG_DIR / '03_rfm_treemap.png',              'NB03 — RFM Customer Segmentation'),
    (FIG_DIR / '03_cohort_retention_heatmap.png', 'NB03 — Cohort Retention Heatmap'),
    (FIG_DIR / '04_rfm_clv_crossvalidation.png',  'NB04 — RFM Segment × CLV Tier Validation'),
]

fig = plt.figure(figsize=(20, 14))
fig.patch.set_facecolor('#1a1a2e')

# Title banner
fig.text(
    0.5, 0.97,
    'Olist E-Commerce Analytics — Executive Dashboard',
    ha='center', va='top', fontsize=20, fontweight='bold',
    color='white', fontfamily='monospace',
)
fig.text(
    0.5, 0.935,
    'RFM Segmentation  ·  Cohort Retention  ·  A/B Experiment  ·  CLV Modelling',
    ha='center', va='top', fontsize=11, color='#aaaacc',
)

gs = gridspec.GridSpec(
    2, 2, figure=fig,
    top=0.92, bottom=0.04,
    left=0.02, right=0.98,
    hspace=0.08, wspace=0.04,
)

for idx, (path, title) in enumerate(DASHBOARD_CHARTS):
    ax = fig.add_subplot(gs[idx // 2, idx % 2])
    ax.set_facecolor('#2a2a3e')
    if path.exists():
        img = mpimg.imread(str(path))
        ax.imshow(img, aspect='auto')
    else:
        ax.text(
            0.5, 0.5,
            f'{title}\n\n(run the corresponding notebook\nto generate this chart)',
            ha='center', va='center', transform=ax.transAxes,
            fontsize=11, color='#888899', linespacing=1.8,
        )
    ax.set_title(title, fontsize=11, fontweight='bold',
                 color='white', pad=6, loc='left')
    ax.axis('off')

plt.savefig(FIG_DIR / '05_executive_dashboard.png', dpi=150,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'Saved → {FIG_DIR / "05_executive_dashboard.png"}')

## 3. Business Findings

The code cell below extracts headline numbers from the loaded tables and
formats them for display. The findings table below that translates those
numbers into plain-English business statements with confidence ratings.
Confidence reflects the strength of the statistical evidence: High means
a clean result with large sample and small p-value; Medium means the result
is directionally clear but based on observational data or a smaller effect.

In [ ]:
# ── Extract headline numbers from loaded tables ────────────────────────────────

def kpi(name):
    if len(kpi_summary) == 0:
        return 'n/a'
    row = kpi_summary[kpi_summary['kpi'] == name]
    return row['value'].iloc[0] if len(row) else 'n/a'

total_revenue   = kpi('Total Revenue')
total_orders    = kpi('Total Orders (delivered)')
unique_custs    = kpi('Unique Customers')
repeat_rate     = kpi('Repeat Purchase Rate')
avg_delivery    = kpi('Avg Delivery Days')
avg_review      = kpi('Avg Review Score')

# Repeat rate from order frequency dist
if len(order_freq) > 0 and 'order_count' in order_freq.columns:
    single_pct = (order_freq.loc[order_freq['order_count'] == 1, 'customer_count'].sum()
                  / order_freq['customer_count'].sum() * 100)
    repeat_pct = 100 - single_pct
else:
    single_pct, repeat_pct = float('nan'), float('nan')

# M1 and M3 retention rates
if len(avg_retention) > 0:
    ret = avg_retention.set_index('cohort_index')['avg_retention_pct']
    m1_ret = f"{ret.get(1, float('nan')):.1f}%" if 1 in ret.index else 'n/a'
    m3_ret = f"{ret.get(3, float('nan')):.1f}%" if 3 in ret.index else 'n/a'
else:
    m1_ret, m3_ret = 'n/a', 'n/a'

# Champions segment
if len(segment_summary) > 0 and 'segment' in segment_summary.columns:
    champ = segment_summary[segment_summary['segment'] == 'Champions']
    champ_pct_cust = f"{champ['pct_customers'].iloc[0]:.1f}%" if len(champ) else 'n/a'
    champ_pct_rev  = f"{champ['pct_revenue'].iloc[0]:.1f}%"   if len(champ) else 'n/a'
else:
    champ_pct_cust, champ_pct_rev = 'n/a', 'n/a'

# Platinum CLV tier
if len(hist_clv_tiers) > 0 and 'tier' in hist_clv_tiers.columns:
    plat = hist_clv_tiers[hist_clv_tiers['tier'] == 'Platinum']
    plat_pct_cust = f"{plat['pct_customers'].iloc[0]:.1f}%" if len(plat) else 'n/a'
    plat_pct_rev  = f"{plat['pct_revenue'].iloc[0]:.1f}%"   if len(plat) else 'n/a'
else:
    plat_pct_cust, plat_pct_rev = 'n/a', 'n/a'

# A/B test — revenue lift
if len(ttest_results) > 0 and 'outcome' in ttest_results.columns:
    rev_row = ttest_results[ttest_results['outcome'] == 'revenue']
    if len(rev_row):
        lift_pct  = f"{rev_row['lift_pct'].iloc[0]:+.1f}%"
        lift_pval = f"{rev_row['p_value'].iloc[0]:.4f}"
        lift_d    = f"{rev_row['cohens_d'].iloc[0]:.3f}"
    else:
        lift_pct = lift_pval = lift_d = 'n/a'
else:
    lift_pct = lift_pval = lift_d = 'n/a'

# Review score gap fast vs slow delivery
if len(review_delivery) > 0 and 'delivery_bin' in review_delivery.columns:
    fast_score = review_delivery.loc[review_delivery['delivery_bin'] == '1-5d', 'mean']
    slow_score = review_delivery.loc[review_delivery['delivery_bin'] == '31-60d', 'mean']
    if len(fast_score) and len(slow_score):
        review_gap = f"{fast_score.iloc[0] - slow_score.iloc[0]:.2f} pts"
    else:
        review_gap = 'n/a'
else:
    review_gap = 'n/a'

# Top category by repeat rate
if len(cat_repeat) > 0 and 'first_category' in cat_repeat.columns:
    top_cat_ret     = cat_repeat.iloc[0]['first_category']
    top_cat_ret_pct = f"{cat_repeat.iloc[0]['repeat_rate_pct']:.1f}%"
else:
    top_cat_ret = top_cat_ret_pct = 'n/a'

print('Key numbers extracted:')
print(f'  Total revenue        : {total_revenue}')
print(f'  Repeat purchase rate : {repeat_rate}')
print(f'  M1 / M3 retention    : {m1_ret} / {m3_ret}')
print(f'  Champions: {champ_pct_cust} of customers, {champ_pct_rev} of revenue')
print(f'  Platinum CLV: {plat_pct_cust} of customers, {plat_pct_rev} of revenue')
print(f'  Free-shipping revenue lift: {lift_pct}  (p={lift_pval}, d={lift_d})')
print(f'  Review gap fast vs slow delivery: {review_gap}')
print(f'  Highest-retention entry category: {top_cat_ret} ({top_cat_ret_pct})')

In [ ]:
# Structured findings table derived from the numbers above
findings = pd.DataFrame([
    {
        'finding': 'Very low repeat purchase rate',
        'metric':  f'Repeat rate ≈ {repeat_rate}',
        'implication': 'Revenue is almost entirely acquisition-driven. '
                       'Small improvements in retention have outsized revenue impact.',
        'confidence': 'High',
        'source': 'NB02',
    },
    {
        'finding': 'Pareto concentration in CLV',
        'metric':  f'Platinum tier ({plat_pct_cust} of customers) generates {plat_pct_rev} of revenue',
        'implication': 'Marketing spend should be heavily weighted toward '
                       'identifying and retaining top-tier customers.',
        'confidence': 'High',
        'source': 'NB04',
    },
    {
        'finding': 'Steep retention drop after first purchase',
        'metric':  f'M1 retention {m1_ret}, M3 retention {m3_ret}',
        'implication': 'The critical window for converting first-time buyers is within '
                       '30 days. Post-purchase email sequences should launch immediately.',
        'confidence': 'High',
        'source': 'NB03',
    },
    {
        'finding': 'Free shipping associated with revenue lift',
        'metric':  f'Lift {lift_pct} (p={lift_pval}, Cohen\'s d={lift_d})',
        'implication': 'Effect is directionally positive but from observational data. '
                       'A randomised trial with an order-value threshold is needed.',
        'confidence': 'Medium',
        'source': 'NB04',
    },
    {
        'finding': 'Delivery speed materially affects satisfaction',
        'metric':  f'Review score gap fast vs slow delivery: {review_gap}',
        'implication': 'Logistics quality is a direct lever on customer satisfaction '
                       'and downstream repeat purchase probability.',
        'confidence': 'High',
        'source': 'NB02',
    },
    {
        'finding': 'Entry category predicts customer loyalty',
        'metric':  f'Top category by repeat rate: {top_cat_ret} ({top_cat_ret_pct})',
        'implication': 'Acquisition campaigns targeting high-retention entry categories '
                       'produce customers with structurally higher LTV.',
        'confidence': 'Medium',
        'source': 'NB03',
    },
    {
        'finding': 'RFM segments align with CLV tiers',
        'metric':  f'Champions: {champ_pct_cust} of customers, {champ_pct_rev} of revenue',
        'implication': 'RFM segmentation is validated as a proxy for CLV and can be '
                       'used operationally without running the full probabilistic model.',
        'confidence': 'High',
        'source': 'NB04',
    },
])

findings.to_csv(TBL_DIR / '05_business_findings.csv', index=False)
print(f'Saved → {TBL_DIR / "05_business_findings.csv"}')
findings

| # | Finding | Metric | Confidence | Source |
|---|---|---|---|---|
| 1 | Very low repeat purchase rate | ~3–5% of customers return | High | NB02 |
| 2 | Top CLV tier (Platinum) dominates revenue | ~25% of customers → ~50% of revenue | High | NB04 |
| 3 | Retention drops sharply after first purchase | M1 ≈ 3–5%, M3 ≈ 1–2% | High | NB03 |
| 4 | Free shipping associated with revenue lift | Positive lift, observational evidence | Medium | NB04 |
| 5 | Faster delivery → higher review scores | ~0.5–1.0 pt gap fast vs slow | High | NB02 |
| 6 | Entry category predicts lifetime loyalty | Top categories show 2–3× avg repeat rate | Medium | NB03 |
| 7 | RFM segments align with CLV tiers | Champions cluster in Platinum CLV | High | NB04 |

## 4. Business Recommendations

The four recommendations below follow directly from the findings above. Each
one is framed as a concrete next action, tied to a specific finding, with a
suggested success metric and the analysis that supports it.

---

### Recommendation 1 — Launch a CLV-Tiered Retention Programme

**Supported by:** Findings 2 and 3 — revenue concentration in Platinum and steep post-purchase retention drop.

The Platinum tier generates a disproportionate share of revenue from a small share of customers. These customers are currently treated identically to Bronze customers in terms of post-purchase communication. A four-tier programme with differentiated touchpoints would protect the most valuable customers and activate the next tier.

| Tier | Proposed action | Trigger |
|---|---|---|
| Platinum | Dedicated account manager, early access, personalised offers | CLV score updated monthly |
| Gold | Cross-category recommendations, loyalty reward at purchase 3 | After second order |
| Silver | Automated win-back email at 30, 60, 90-day inactivity | No order in 30 days |
| Bronze | Low-cost single win-back SMS at 60-day inactivity | No order in 60 days |

**Success metric:** 12-month revenue per customer in Gold tier vs. control cohort.

---

### Recommendation 2 — Run a Properly Randomised Free-Shipping Experiment

**Supported by:** Finding 4 — free shipping shows a positive revenue association, but the current evidence is observational.

The observational analysis cannot confirm causality because customers who receive free shipping may differ systematically from those who do not. The regression-adjusted estimate reduces but does not eliminate this bias. The right next step is a clean randomised trial with a minimum order threshold to protect margin.

**Proposed design:**
- **Control:** Standard shipping for all eligible orders
- **Treatment:** Free shipping on orders ≥ R$120 in the top three high-LTV entry categories
- **Randomisation:** Customer-level (not order-level) to avoid contamination
- **Primary metric:** 90-day CLV, not single-order AOV
- **Required n:** ≥ 3,000 per group (use `compute_sample_size` with d=0.20 for a 10% AOV lift)
- **Duration:** Minimum 4 weeks, avoiding promotional calendar events

**Success metric:** 90-day CLV uplift in treatment group vs. control, net of shipping cost.

---

### Recommendation 3 — Shift Acquisition Budget Toward High-Retention Entry Categories

**Supported by:** Finding 6 — first purchase category is a strong predictor of repeat purchase rate and lifetime value.

Not all acquisition spend is equal. Customers acquired through high-retention entry categories have structurally higher LTV and pay back acquisition cost faster. Reallocating a portion of paid acquisition budget toward these category landing pages — and measuring the result with a Category LTV Index — would improve marketing ROI without requiring a headcount increase.

**Proposed actions:**
1. Identify the top five entry categories by repeat rate (minimum 200 customers, from `03_category_repeat_rate.csv`)
2. Increase paid search and social bids on those category pages by 20–30%
3. Track **Category LTV Index** = avg 12-month CLV of customers acquired via that category / overall avg CLV
4. Review monthly; reallocate quarterly based on observed index scores

**Success metric:** Blended 12-month CLV of newly acquired customers, measured quarterly.

---

### Recommendation 4 — Set a Delivery SLA and Instrument It as a Business KPI

**Supported by:** Finding 5 — faster delivery is strongly associated with higher review scores, and review scores correlate with repeat purchase probability.

Delivery experience is the single post-purchase touchpoint Olist controls most directly. The review score gap between fast and slow delivery is large enough to be commercially meaningful. Two changes would move the needle: setting an explicit late-delivery SLA with sellers, and fixing estimated delivery dates (many 'late' flags arise from inaccurate estimates, not genuinely slow fulfilment).

**Proposed actions:**
1. Define late delivery as actual delivery > estimated date and measure the rate weekly
2. Target late delivery rate below 10% as a seller-facing SLA
3. Flag sellers with persistent late delivery rates > 20% for review
4. Add **Avg Review Score** and **On-Time Delivery Rate** to the weekly business review dashboard alongside revenue and order volume

**Success metric:** Average review score ≥ 4.2 / 5 within two quarters of SLA implementation.